# Load one single shot file

Dependencies: Python 3.9+, NumPy, h5py, Matplotlib and JupyterLab. If needed, run `%pip install numpy h5py matplotlib` in a notebook cell. Both notebooks are self-contained; no other scripts or inventory files are needed.

Run top to bottom. Change `FILE_PATH` and choose whether to plot. Data files are never modified.

In [ ]:
from pathlib import Path
import re
import numpy as np
import h5py
import matplotlib.pyplot as plt

NUMBER = re.compile(r'[-+]?(?:(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?|inf(?:inity)?|nan)', re.I)

def decode_value(value):
    """Decode bytes and complete serialized vectors; preserve other text verbatim.

    Raise on abbreviated arrays rather than silently treating '...' as full data.
    Nested arrays remain original text, accessible through raw_fields.
    """
    if isinstance(value, np.ndarray):
        if value.dtype.kind not in 'SUO':
            return value.copy()
        return [decode_value(x) for x in value.tolist()]
    if isinstance(value, (bytes, np.bytes_)):
        value = bytes(value).decode('utf-8', errors='strict')
    if isinstance(value, str):
        text = value.strip()
        if text == 'None':
            return None
        if text.startswith('[') and text.endswith(']'):
            if '...' in text:
                raise ValueError('Serialized array contains ellipsis; original samples are unavailable')
            if '[' in text[1:-1]:
                return value
            remainder = NUMBER.sub('', text[1:-1])
            if remainder.replace(',', '').strip():
                raise ValueError('Unrecognized content in serialized numeric vector')
            return np.fromstring(text[1:-1].replace(',', ' '), sep=' ', dtype=float)
        return value
    if isinstance(value, np.generic):
        return value.item()
    return value

def read_file(path):
    """Read all populated records in one HDF5 file, keeping field names unchanged."""
    records = []
    with h5py.File(path, 'r') as handle:
        for qubit, group in handle.items():
            if not isinstance(group, h5py.Group) or 'Dates' not in group:
                continue
            dates = np.atleast_1d(group['Dates'][()])
            for index, date in enumerate(dates):
                if not np.isfinite(float(date)):
                    continue
                fields = {}
                for key, dataset in group.items():
                    if not isinstance(dataset, h5py.Dataset):
                        continue
                    raw = dataset[()]
                    if isinstance(raw, np.ndarray) and raw.ndim and raw.shape[0] == len(dates):
                        raw = raw[index]
                    fields[key] = decode_value(raw)
                records.append(dict(source_file=str(path), qubit=qubit,
                                    dataset_index=index, fields=fields))
    if not records:
        raise ValueError(f'No populated records: {path}')
    return records

def plot_single_shot(record):
    """Plot every stored IQ sample and unrotated I histograms; return the figure.

    Ground/excited labels describe preparation. No reclassification, rotation,
    fidelity fitting, averaging or point subsampling is performed.
    """
    import matplotlib.pyplot as plt
    fields = record['fields']
    for state in ('g', 'e'):
        for axis in ('I', 'Q'):
            value = fields.get(f'{axis}_{state}')
            if not isinstance(value, np.ndarray) or value.ndim != 1 or not value.size:
                raise ValueError('Plotting requires single-shot I_g, Q_g, I_e and Q_e vectors.')
        if fields[f'I_{state}'].size != fields[f'Q_{state}'].size:
            raise ValueError(f'{state} I/Q lengths differ')
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), constrained_layout=True)
    for state, label, color in [('g', 'Prepared ground', '#2463a7'),
                                ('e', 'Prepared excited', '#ce5935')]:
        i, q = fields[f'I_{state}'], fields[f'Q_{state}']
        axes[0].scatter(i, q, s=3, alpha=.2, color=color, rasterized=True,
                        label=f'{label} (n={len(i):,})')
        axes[1].hist(i[np.isfinite(i)], bins=80, histtype='step', linewidth=1.6,
                     color=color, label=label)
    axes[0].set(xlabel='I (stored units)', ylabel='Q (stored units)', title='Raw single-shot IQ')
    axes[0].legend(markerscale=3, fontsize=8)
    axes[1].set(xlabel='I (stored units, unrotated)', ylabel='Shots per bin', title='I distributions')
    axes[1].legend(fontsize=8)
    for ax in axes:
        ax.grid(alpha=.15)
    fidelity = fields.get('Fidelity')
    suffix = f' | stored SSF contrast {fidelity:.4f}' if isinstance(fidelity, (float, int)) else ''
    fig.suptitle(f"{record['qubit']} | record {record['dataset_index']}{suffix}")
    return fig

In [ ]:
# Set PACKAGE_ROOT if Jupyter starts outside the extracted package.
PACKAGE_ROOT = None
if PACKAGE_ROOT is None:
    PACKAGE_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                         if (p / 'datasets').is_dir() and (p / 'documentation').is_dir()), None)
    if PACKAGE_ROOT is None:
        raise FileNotFoundError('Set PACKAGE_ROOT to your extracted package folder.')
PACKAGE_ROOT = Path(PACKAGE_ROOT).expanduser().resolve()


## Choose the file and plotting option

Use an absolute path or a path relative to the package folder. Qubit labels such as Q2 are one-based; folder qubit_1round0 corresponds to Q2. `DATASET_INDEX` is zero-based. `PLOT=False` loads arrays without making a plot.

In [ ]:
FILE_PATH = str(next(iter(sorted((PACKAGE_ROOT / 'datasets').glob('*/qubit*round*/single_shot/data/*.h5')))).relative_to(PACKAGE_ROOT))
QUBIT = None
DATASET_INDEX = 0
PLOT = True
SAVE_PLOT = False

path = Path(FILE_PATH).expanduser()
if not path.is_absolute():
    path = PACKAGE_ROOT / path
records = [r for r in read_file(path) if (QUBIT is None or r['qubit'] == QUBIT)
           and r['dataset_index'] == DATASET_INDEX]
if len(records) != 1:
    raise ValueError('Choose a populated QUBIT and DATASET_INDEX that select exactly one record.')
record = records[0]
f = record['fields']
I_g, Q_g, I_e, Q_e = (f[k] for k in ['I_g', 'Q_g', 'I_e', 'Q_e'])
print('File:', path.name)
print('Qubit:', record['qubit'], '| Stored SSF contrast:', f.get('Fidelity'))
print('IQ shapes:', I_g.shape, Q_g.shape, I_e.shape, Q_e.shape)
if PLOT or SAVE_PLOT:
    fig = plot_single_shot(record)
    if SAVE_PLOT:
        fig.savefig(PACKAGE_ROOT / 'selected_single_shot.png', dpi=160)
    if PLOT:
        plt.show()
    else:
        plt.close(fig)

The arrays above retain all samples and their original order. Plotting shows raw I/Q and unrotated I histograms; it does not refit fidelity or apply the stored Angle. To load a calibration file, use `read_file(path)` and inspect its `fields`; the single-shot plot helper requires I_g, Q_g, I_e and Q_e.